# Fake News Detection — Task 6: Explainability Module

---

## Overview

This notebook implements and demonstrates the **Step 6 Explainability Module**.

It does **not** retrain anything. It uses the SAME artifacts saved in Steps 3 & 4:
- `models/best_model.pkl`
- `models/tfidf_vectorizer.pkl`

Three levels of explanation are provided:

| Level | What it shows |
|---|---|
| **Global feature importance** | Which words the model globally associates with FAKE/REAL |
| **Article-specific explanation** | Which words *in this article* drove the prediction |
| **Suspicious language detection** | Clickbait/sensational linguistic patterns in the raw text |

> **Important:** These explanations describe the **model's decision**, not factual truth.
> A word appearing in the influential features list does **not** prove an article is fake.

---

## Flow

```
News Text
    ↓
Step 5 Prediction Pipeline  →  FAKE / REAL + Confidence
    ↓
Step 6 Explainability       →  Influential Features + Suspicious Language
```

---

## Section 1: Setup — Import Libraries and Modules

In [ ]:
import sys, os
import warnings
warnings.filterwarnings('ignore')

# Add src/ to path so we can import our modules
sys.path.insert(0, '../src')

# ── Import Step 5 prediction module (unchanged) ──
import prediction as pred

# ── Import Step 6 explainability module ──
import explainability as exp

print('Modules loaded successfully!')
print('Model type    :', pred.pipeline_info()['model_type'])
print('Vocabulary    :', pred.pipeline_info()['vectorizer_vocab_size'], 'features')
print('Label mapping :', pred.pipeline_info()['label_mapping'])

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

print('Visualization libraries imported.')

---

## Section 2: Inspect the Loaded Model

Before explaining predictions, we confirm the model type and verify that
coefficient-based explainability is available.

For a binary classifier with labels 0/1:
- `coef_[0]` is the decision boundary normal vector.
- **Positive coefficient** → word pushes prediction toward `classes_[-1]` = 1 = **REAL**
- **Negative coefficient** → word pushes prediction toward `classes_[0]`  = 0 = **FAKE**

In [ ]:
info = exp.model_info()
print('=' * 50)
print('  MODEL INTROSPECTION')
print('=' * 50)
for k, v in info.items():
    print(f'  {k:26s}: {v}')
print()

if info['has_coef']:
    print(f'  Explainability method: {info["explainability_method"]}')
    print(f'  Coefficient direction:')
    print(f'    positive coef  →  pushes toward REAL (class 1)')
    print(f'    negative coef  →  pushes toward FAKE (class 0)')
else:
    print('  WARNING: model does not expose coef_.')
    print('  Article-specific coefficient-based explanation unavailable.')

In [ ]:
# Confirm model.classes_ matches our label mapping
classes = pred.model.classes_
print('model.classes_       :', classes)
print('classes_[0]  = FAKE  :', pred.LABEL_NAMES[classes[0]])
print('classes_[-1] = REAL  :', pred.LABEL_NAMES[classes[-1]])
print()
print('Coefficient vector shape:', pred.model.coef_.shape)
print('(1 row = 1 decision boundary for binary classification)')

---

## Section 3: Global Feature Importance

These are the words/bigrams the model has **globally** learned to associate with FAKE or REAL news.

| What it is | How it's computed |
|---|---|
| Top REAL features | Words with the **highest positive** model coefficients |
| Top FAKE features | Words with the **most negative** model coefficients |

> This reflects the full training corpus, not any specific article.

In [ ]:
importance = exp.get_feature_importance(top_n=20)

print('=' * 60)
print('  TOP 20 FEATURES → REAL NEWS (highest positive coefficients)')
print('=' * 60)
for i, feat in enumerate(importance['top_real_features'], 1):
    print(f'  {i:2d}. {feat["word"]:35s}  coef = {feat["coefficient"]:+.4f}')

print()
print('=' * 60)
print('  TOP 20 FEATURES → FAKE NEWS (most negative coefficients)')
print('=' * 60)
for i, feat in enumerate(importance['top_fake_features'], 1):
    print(f'  {i:2d}. {feat["word"]:35s}  coef = {feat["coefficient"]:+.4f}')

print()
print('Note:', importance['note'])

In [ ]:
# ── Visualize global feature importance ──
real_words  = [f['word'] for f in importance['top_real_features'][:15]]
real_coefs  = [f['coefficient'] for f in importance['top_real_features'][:15]]
fake_words  = [f['word'] for f in importance['top_fake_features'][:15]]
fake_coefs  = [f['coefficient'] for f in importance['top_fake_features'][:15]]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))
fig.suptitle('Global Feature Importance (Model Coefficients)', fontsize=14, fontweight='bold')

ax1.barh(real_words[::-1], real_coefs[::-1], color='#27ae60', alpha=0.85, edgecolor='white')
ax1.set_title('Top 15 → REAL NEWS', fontsize=12, color='#27ae60', fontweight='bold')
ax1.set_xlabel('Model Coefficient')
ax1.axvline(0, color='black', linewidth=0.8)
ax1.grid(axis='x', alpha=0.3)

ax2.barh(fake_words, fake_coefs, color='#e74c3c', alpha=0.85, edgecolor='white')
ax2.set_title('Top 15 → FAKE NEWS', fontsize=12, color='#e74c3c', fontweight='bold')
ax2.set_xlabel('Model Coefficient')
ax2.axvline(0, color='black', linewidth=0.8)
ax2.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.savefig('global_feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()
print('Chart saved as global_feature_importance.png')

---

## Section 4: Article-Specific Explanation — `explain_prediction()`

For a specific article, contribution is computed as:

```
contribution[i]  =  tfidf_score[i]  ×  coef[i]
```

- `tfidf_score[i]` — how strongly the i-th word appears in *this* article.
- `coef[i]`        — model's global association with REAL (positive) or FAKE (negative).
- Only words that actually appear in the article (non-zero TF-IDF) are considered.

Features are ranked by **absolute contribution** to show the most impactful words first.

In [ ]:
# Quick demonstration — article-specific explanation
demo_text = (
    'WASHINGTON (Reuters) - The Federal Reserve on Wednesday raised its benchmark '
    'interest rate by a quarter percentage point, citing continued strength in the '
    'labor market and persistent inflation pressures. Fed Chair Jerome Powell said '
    'the central bank remains committed to its two percent inflation target over the '
    'medium term, following the two-day policy meeting held in Washington.'
)

demo_result = exp.explain_prediction(demo_text, top_n=10)

print('Prediction  :', demo_result['prediction'])
print('Confidence  :', demo_result['confidence'], '%')
print()
print(f'{"Rank":<5}  {"Word":<32}  {"TF-IDF":>8}  {"Coef":>8}  {"Contribution":>12}  Direction')
print('-' * 80)
for i, feat in enumerate(demo_result['influential_features'], 1):
    print(f'{i:<5}  {feat["word"]:<32}  {feat["tfidf_score"]:>8.4f}  '
          f'{feat["coefficient"]:>8.4f}  {feat["contribution"]:>12.4f}  {feat["direction"]}')

---

## Section 5: Suspicious / Clickbait Language Detection

A small, transparent list of common clickbait and sensational linguistic patterns
is matched against the **raw** (unprocessed) text using regex.

These are categorised as:

| Category | Examples |
|---|---|
| sensational urgency | BREAKING, URGENT, BOMBSHELL |
| clickbait call-to-action | SHARE THIS NOW, MAKE THIS VIRAL |
| clickbait framing | "you won't believe", "before it's deleted" |
| conspiracy language | deep state, Big Pharma, secret cure |
| exaggerated claim | 100% proven, GUARANTEED, miracle cure |
| emotional manipulation | SHOCKING, DEVASTATING, OUTRAGEOUS |

> **These patterns are heuristic indicators only.** Their presence does not mean an article is fake.
> Many legitimate journalism outlets use urgent language. Always treat this list as supplementary context.

In [ ]:
# Test suspicious language detection on two example texts
clean_article = (
    'The Federal Reserve raised interest rates by 0.25 percentage points on Wednesday, '
    'as widely expected by market analysts following two days of policy discussions.'
)

sensational_article = (
    'BOMBSHELL REVELATION! Scientists HATE this! You won\'t believe what Big Pharma '
    'has been hiding for years. Share this now before it gets deleted! '
    'GUARANTEED 100% proven miracle cure — MAKE THIS VIRAL!'
)

print('=== Clean article — suspicious patterns ===')
findings_clean = exp._detect_suspicious_language(clean_article)
if findings_clean:
    for f in findings_clean:
        print(f'  [{f["category"]}]  {f["pattern_desc"]}  (matched: "{f["matched_text"]}")')
else:
    print('  No obvious suspicious language detected.')

print()
print('=== Sensational article — suspicious patterns ===')
findings_sens = exp._detect_suspicious_language(sensational_article)
if findings_sens:
    for f in findings_sens:
        print(f'  [{f["category"]}]  {f["pattern_desc"]}  (matched: "{f["matched_text"]}")')
else:
    print('  No obvious suspicious language detected.')

---

## Section 6: Test Examples

We test the full explainability module with three examples:

| Test | Input type | Expected |
|---|---|---|
| Test 1 | Formal Reuters-style article | REAL (possibly) |
| Test 2 | Sensational clickbait-style article | FAKE (possibly) |
| Test 3 | Invalid / empty input | Error dict |

> The predictions are driven by the model's learned patterns.
> Results may vary — what matters is that the explanation is correct and consistent.

In [ ]:
# ── Helper: formatted display ──
def print_explanation(result):
    exp.display_explanation(result, show_direction_split=True)

print('Display helper defined.')

In [ ]:
# ── TEST 1: Formal news article (REAL-leaning) ──
test_article_1 = (
    'WASHINGTON (Reuters) - The Federal Reserve on Wednesday raised its benchmark '
    'interest rate by a quarter percentage point for the fourth time this year, '
    'citing continued strength in the labor market and persistent inflation pressures. '
    'Fed Chair Jerome Powell said in a statement that the central bank remains '
    'committed to bringing inflation back to its two percent target over the medium '
    'term. The decision was widely expected by financial markets following the '
    'two-day policy meeting held in Washington this week.'
)

print('TEST 1 — Formal news article')
result_1 = exp.get_explanation(test_article_1, top_n=10)
print_explanation(result_1)

In [ ]:
# ── TEST 2: Sensational clickbait-style article (FAKE-leaning) ──
test_article_2 = (
    'BOMBSHELL REVELATION: You won\'t believe what scientists just discovered! '
    'This SHOCKING secret the government doesn\'t want you to know will change '
    'everything you thought you knew. Share this now before it gets deleted! '
    'Experts are furious about this one simple GUARANTEED miracle cure that '
    'Big Pharma has been hiding for years. MAKE THIS VIRAL! 100% proven effective.'
)

print('TEST 2 — Sensational clickbait-style article')
result_2 = exp.get_explanation(test_article_2, top_n=10)
print_explanation(result_2)

In [ ]:
# ── TEST 3: Invalid / empty input ──
print('TEST 3 — Invalid/empty input')
result_3 = exp.get_explanation('')
print_explanation(result_3)
assert 'error' in result_3, 'Expected error dict for empty input!'
print('Validation correctly rejected empty input without crashing.')

---

## Section 7: Visualization — Article-Specific Feature Contributions

A horizontal bar chart showing the top influential features for a specific article.

- **Green bars** → features pushing the prediction toward **REAL**
- **Red bars** → features pushing the prediction toward **FAKE**
- Bar length = absolute contribution magnitude

In [ ]:
def plot_explanation(result, title='Article-Specific Feature Contributions', save_as=None):
    """
    Horizontal bar chart of the top influential features for one article.
    Green = pushing toward REAL, Red = pushing toward FAKE.
    """
    if 'error' in result:
        print('Cannot plot — result contains an error:', result['error'])
        return

    features = result.get('influential_features', [])
    if not features:
        print('No features to plot.')
        return

    words     = [f['word'] for f in features]
    contribs  = [f['contribution'] for f in features]
    colors    = ['#27ae60' if c >= 0 else '#e74c3c' for c in contribs]

    fig, ax = plt.subplots(figsize=(10, max(4, len(words) * 0.45)))

    bars = ax.barh(words[::-1], contribs[::-1], color=colors[::-1],
                   alpha=0.87, edgecolor='white', linewidth=0.6)

    ax.axvline(0, color='black', linewidth=1.2, linestyle='--')
    ax.set_xlabel('Contribution  (tfidf_score × model_coefficient)', fontsize=10)
    ax.set_title(
        f'{title}\n'
        f'Prediction: {result["prediction"]}  |  Confidence: {result["confidence"]:.1f}%',
        fontsize=11, fontweight='bold'
    )
    ax.grid(axis='x', alpha=0.3)

    real_patch = mpatches.Patch(color='#27ae60', alpha=0.87, label='Pushes toward REAL')
    fake_patch = mpatches.Patch(color='#e74c3c', alpha=0.87, label='Pushes toward FAKE')
    ax.legend(handles=[real_patch, fake_patch], fontsize=9, loc='lower right')

    plt.tight_layout()
    if save_as:
        plt.savefig(save_as, dpi=150, bbox_inches='tight')
        print(f'Chart saved as {save_as}')
    plt.show()

print('plot_explanation() function defined.')

In [ ]:
# Plot for Test 1 (formal article)
plot_explanation(result_1, title='Test 1 — Formal News Article', save_as='explanation_test1.png')

In [ ]:
# Plot for Test 2 (sensational article)
plot_explanation(result_2, title='Test 2 — Sensational Article', save_as='explanation_test2.png')

---

## Section 8: Combined `get_explanation()` — Structured Output

The main public function `get_explanation(text)` returns a single clean dictionary
that the future Streamlit app can consume directly:

```python
from explainability import get_explanation
result = get_explanation('some raw article text...')
```

Output structure:

```python
{
    'prediction'          : 'FAKE' | 'REAL',
    'confidence'          : float,
    'influential_features': [...],
    'features_for_real'   : [...],
    'features_for_fake'   : [...],
    'suspicious_language' : [...],
    'note'                : str
}
```

In [ ]:
# ── Show the raw dict structure ──
import pprint

sample_text = (
    'A congressional committee launched an investigation Tuesday into the '
    'administration\'s handling of classified documents, with bipartisan '
    'support from both Republican and Democratic members of the committee. '
    'The chairman said testimony from senior officials would be sought '
    'within the next two weeks, according to a statement released by the panel.'
)

structured_result = exp.get_explanation(sample_text, top_n=8)

# Show the keys present
print('Keys in result dict:')
for k in structured_result:
    val = structured_result[k]
    if isinstance(val, list):
        print(f'  {k}: list of {len(val)} item(s)')
    else:
        print(f'  {k}: {repr(val)[:80]}')

In [ ]:
# ── Show influential_features in full ──
print('influential_features:')
pprint.pprint(structured_result['influential_features'])

In [ ]:
# ── Show suspicious_language in full ──
suspicious = structured_result['suspicious_language']
print('suspicious_language:')
if suspicious:
    pprint.pprint(suspicious)
else:
    print('  []  (none detected)')

---

## Section 9: Final Verification

Confirm that all requirements from the Step 6 spec are satisfied.

In [ ]:
import joblib

print('=' * 55)
print('  STEP 6 — FINAL VERIFICATION')
print('=' * 55)
print()

# ── 1. Saved model loads ─────────────────────────────────────────
assert pred.model is not None, 'Model did not load!'
print('  [OK] Saved model loads successfully.')
print(f'       Type: {type(pred.model).__name__}')

# ── 2. Saved vectorizer loads ────────────────────────────────────
assert pred.vectorizer is not None, 'Vectorizer did not load!'
print('  [OK] Saved TF-IDF vectorizer loads successfully.')
print(f'       Vocabulary size: {len(pred.vectorizer.vocabulary_):,}')

# ── 3. Same preprocessing from Step 2/5 is used ──────────────────
import inspect
src_exp  = inspect.getsource(exp._preprocess)
src_pred = inspect.getsource(pred.preprocess_text)
assert src_exp == src_pred, 'preprocess_text MISMATCH between modules!'
print('  [OK] preprocess_text() is identical in explainability and prediction modules.')

# ── 4. No retraining ─────────────────────────────────────────────
# No .fit() call exists in explainability.py — this is verified by inspection.
exp_source = inspect.getsource(exp)
assert '.fit(' not in exp_source, 'fit() call found in explainability module!'
print('  [OK] No model retraining (no .fit() call in explainability module).')

# ── 5. Correct class mapping ──────────────────────────────────────
assert pred.LABEL_NAMES[0] == 'FAKE', 'Label mapping wrong!'
assert pred.LABEL_NAMES[1] == 'REAL', 'Label mapping wrong!'
print('  [OK] Correct label mapping confirmed: 0=FAKE, 1=REAL.')

# ── 6. Feature names from actual vectorizer ───────────────────────
feat_names = exp._get_feature_names()
assert len(feat_names) > 0, 'Feature names empty!'
assert len(feat_names) == len(pred.vectorizer.vocabulary_), 'Size mismatch!'
print(f'  [OK] Feature names come from actual TF-IDF vectorizer ({len(feat_names):,} features).')

# ── 7. Explanation based on actual model ─────────────────────────
coef = exp._get_coef()
assert coef is not None, 'Coef is None — cannot explain!'
assert coef.shape[0] == len(feat_names), 'Coef/feature-name size mismatch!'
print(f'  [OK] Explanation uses actual model.coef_ (shape: {coef.shape}).')

# ── 8. Invalid input handled safely ──────────────────────────────
for bad in ['', '   ', None, 'ok']:
    r = exp.get_explanation(bad)
    assert 'error' in r, f'Expected error for input {repr(bad)}'
print('  [OK] Invalid inputs return clean error dicts, no stack traces.')

# ── 9. Required output keys present ──────────────────────────────
verification_text = (
    'Some realistic news article text about the economy and government policy. '
    'The president announced a new initiative aimed at reducing inflation and '
    'supporting small businesses across the country.'
)
r = exp.get_explanation(verification_text)
for key in ('prediction', 'confidence', 'influential_features'):
    assert key in r, f'Missing required key: {key}'
assert r['prediction'] in ('FAKE', 'REAL'), 'Invalid prediction value!'
assert isinstance(r['confidence'], (int, float)), 'Confidence must be numeric!'
assert isinstance(r['influential_features'], list), 'influential_features must be a list!'
print(f'  [OK] get_explanation() returns required structure.')
print(f'       prediction={r["prediction"]}, confidence={r["confidence"]}%')
print(f'       influential_features: {len(r["influential_features"])} items')

print()
print('All 9 verification checks passed!')
print('Step 6 — Explainability Module is complete and verified.')

---

## Task 6 Complete — Summary

### What Was Built

| Component | Location | Description |
|---|---|---|
| `get_feature_importance()` | `src/explainability.py` | Global top FAKE/REAL words from model.coef_ |
| `explain_prediction()` | `src/explainability.py` | Article-specific contribution = tfidf × coef |
| `_detect_suspicious_language()` | `src/explainability.py` | Heuristic clickbait pattern detection |
| `get_explanation()` | `src/explainability.py` | Combined entry point for Streamlit |
| `display_explanation()` | `src/explainability.py` | Pretty-printer for notebook/CLI use |

### Files Created / Modified

| File | Status |
|---|---|
| `src/explainability.py` | **NEW** — Step 6 module |
| `notebooks/Task6_Explainability.ipynb` | **NEW** — this notebook |
| `src/prediction.py` | **UNCHANGED** |
| `models/best_model.pkl` | **UNCHANGED** |
| `models/tfidf_vectorizer.pkl` | **UNCHANGED** |

### Output Structure (for Streamlit)

```python
from explainability import get_explanation

result = get_explanation('raw article text...')
# {
#   'prediction'          : 'FAKE' | 'REAL',
#   'confidence'          : float,
#   'confidence_type'     : str,
#   'model_used'          : str,
#   'influential_features': [{'word', 'tfidf_score', 'coefficient', 'contribution', 'direction'}, ...],
#   'features_for_real'   : [...],
#   'features_for_fake'   : [...],
#   'suspicious_language' : [{'pattern_desc', 'category', 'matched_text'}, ...],
#   'note'                : str
# }
```

### Known Limitation (inherited from Step 5)

The best model is **LinearSVC**, which does not provide calibrated probabilities.
The `confidence` value is a sigmoid-squashed decision margin — a proxy, not a
true probability. This is clearly labeled via `confidence_type`.

### What's Next

The Streamlit app (Step 9) can now call:
```python
from explainability import get_explanation, plot_explanation
result = get_explanation(user_input)
```
and render the prediction, confidence, feature bars, and suspicious language flags
directly from the returned dictionary.